# Sugar Trap: Blue Ocean Market Gap Analysis
### Helix CPG Partners · Open Food Facts Dataset
> **Objective:** Identify under-served "High Protein / Low Sugar" snack categories  

## 0. Setup & Dependencies

In [1]:
import warnings, re, ast
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter
from IPython.display import display, HTML
import kaleido # Explicitly import kaleido

#Drive

from google.colab import drive, auth
import gspread
from gspread_dataframe import set_with_dataframe
from google.auth import default

# ── reproducible seed ──
np.random.seed(42)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.2f}".format)
print("All imports OK")

import plotly.io as pio
pio.renderers.default = "notebook_connected"

All imports OK


In [2]:
# Google Drive mount
drive.mount('/content/drive', force_remount=False)
DATA_FOLDER = '/content/drive/MyDrive/AmalitechProject/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Story 1 · Data Ingestion & "The Clean-Up"
We stream only the columns we need from the Open Food Facts CSV (tab-separated).  
This avoids loading the full 3 GB+ file into memory.

In [3]:
# COLUMNS WE ACTUALLY NEED
# nutriscore_grade is the current column name (nutrition_grade_fr is legacy)
COLS_WANTED = [
    "product_name", "categories_tags", "ingredients_text",
    "sugars_100g", "proteins_100g", "fat_100g",
    "fiber_100g", "energy_100g",
    "nutriscore_grade",   # current name
    "nutrition_grade_fr", # legacy fallback
]

DATA_URL = (
    "https://static.openfoodfacts.org/data/"
    "en.openfoodfacts.org.products.csv.gz"
)

# ─Load first 500 000 rows — NO usecols so we never crash on missing cols ──
CHUNK  = 50_000
N_ROWS = 500_000

chunks = []
try:
    reader = pd.read_csv(
        DATA_URL,
        sep="\t",
        nrows=N_ROWS,
        low_memory=False,
        chunksize=CHUNK,
        compression="infer",
        on_bad_lines="skip",
        encoding="utf-8",
    )
    for i, chunk in enumerate(reader):
        # Keep only columns we want that actually exist in this file
        keep = [c for c in COLS_WANTED if c in chunk.columns]
        chunks.append(chunk[keep])
        print(f"   chunk {i+1:>2} — {len(chunk):,} rows", end="\r")
    raw = pd.concat(chunks, ignore_index=True)
    print(f"\n Loaded from .gz")
except Exception as e:
    print(f"gz failed ({e}), trying plain TSV …")
    raw = pd.read_csv(
        "https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv",
        sep="\t", nrows=N_ROWS, low_memory=False,
        on_bad_lines="skip", encoding="utf-8",
    )
    keep = [c for c in COLS_WANTED if c in raw.columns]
    raw = raw[keep]

# Normalise nutriscore column name
if 'nutrition_grade_fr' in raw.columns and 'nutriscore_grade' not in raw.columns:
    raw = raw.rename(columns={'nutrition_grade_fr': 'nutriscore_grade'})
elif 'nutriscore_grade' not in raw.columns:
    raw['nutriscore_grade'] = None  # column missing entirely — create empty

print(f"\n Raw shape: {raw.shape}")
print(f"Columns loaded: {raw.columns.tolist()}")
raw.head()


   chunk 10 — 50,000 rows
 Loaded from .gz

 Raw shape: (500000, 9)
Columns loaded: ['product_name', 'categories_tags', 'ingredients_text', 'sugars_100g', 'proteins_100g', 'fat_100g', 'fiber_100g', 'energy_100g', 'nutriscore_grade']


,product_name,categories_tags,ingredients_text,sugars_100g,proteins_100g,fat_100g,fiber_100g,energy_100g,nutriscore_grade
0,Limonade artisanale a la rose,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unknown
1,M&amp;M white,NaN,"Weizenmehl, Rapsöl, Speisesalz, 1,7% Meersalz,...",NaN,NaN,NaN,NaN,NaN,unknown
2,Chocolate n3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unknown
3,Pâte de fruits,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unknown
4,Paleta gran reserva - Sierra nevada-,"en:beverages-and-beverages-preparations,en:bev...","Thiamin, Biotin, Chromium, Garcinia cambogia f...",NaN,NaN,NaN,NaN,NaN,unknown


### 1b. Data Cleaning
**Rules applied:**
- Drop rows missing `product_name`, `sugars_100g`, or `proteins_100g`
- Remove biologically impossible values (nutrients > 100 g per 100 g, or < 0)
- Cap `energy_100g` at 4,000 kcal (pure fat ceiling)
- Drop exact duplicates on `product_name`

In [4]:

def clean_food_facts(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Drop mandatory nulls
    df.dropna(subset=["product_name", "sugars_100g", "proteins_100g"], inplace=True)
    df["product_name"] = df["product_name"].str.strip()
    df = df[df["product_name"] != ""]

    # 2. Numeric coercion
    for col in ["sugars_100g", "proteins_100g", "fat_100g",
                "fiber_100g", "energy_100g"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 3. Physiological bounds
    for col in ["sugars_100g", "proteins_100g", "fat_100g", "fiber_100g"]:
        df = df[(df[col] >= 0) & (df[col] <= 100)]
    df = df[(df["energy_100g"] >= 0) & (df["energy_100g"] <= 4_000)]

    # 4. Macro-sum sanity (sugars+proteins+fat <= 105 to allow rounding)
    macro_sum = df["sugars_100g"] + df["proteins_100g"] + df["fat_100g"].fillna(0)
    df = df[macro_sum <= 105]

    # 5. De-duplicate
    df.drop_duplicates(subset="product_name", inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

df = clean_food_facts(raw)

print(f"Raw rows  : {len(raw):>8,}")
print(f"Clean rows: {len(df):>8,}  ({len(df)/len(raw)*100:.1f}% retained)")
display(df.describe())


Raw rows  :  500,000
Clean rows:   64,057  (12.8% retained)


,sugars_100g,proteins_100g,fat_100g,fiber_100g,energy_100g
count,64057.00,64057.00,64057.00,64057.00,64057.00
mean,12.37,8.55,11.61,2.94,1164.57
std,17.59,9.56,14.50,5.10,746.64
min,0.00,0.00,0.00,0.00,0.00
25%,1.18,2.60,1.18,0.00,485.00
50%,4.42,6.67,5.80,1.60,1133.86
75%,15.79,10.87,17.86,3.57,1715.28
max,100.00,100.00,100.00,100.00,3940.93


## Story 2 · The Category Wrangler
We parse the `categories_tags` string and assign each product to one of  
**8 high-level buckets** using keyword priority matching.

In [5]:
# Keyword priority map (first match wins)
CATEGORY_RULES = [
    ("Protein Bars & Shakes",  ["protein-bar", "protein-shake", "protein-powder",
                                 "meal-replacement", "sport", "fitness"]),
    ("Nuts & Seeds",           ["nut", "seed", "almond", "cashew", "peanut",
                                 "pistachio", "walnut", "sunflower-seed"]),
    ("Dairy & Eggs",           ["dairy", "cheese", "yogurt", "yoghurt", "milk",
                                 "egg", "butter", "cream"]),
    ("Cereals & Granola",      ["cereal", "granola", "oat", "muesli", "breakfast"]),
    ("Chips & Savory Snacks",  ["chip", "crisp", "popcorn", "pretzel", "cracker",
                                 "savoury-snack", "savory-snack", "corn-snack"]),
    ("Candy & Confectionery",  ["candy", "chocolate", "confectionery", "sweet",
                                 "gummy", "marshmallow", "caramel", "lollipop"]),
    ("Baked Goods & Cookies",  ["cookie", "biscuit", "cake", "pastry", "muffin",
                                 "brownie", "wafer", "cracker"]),
    ("Fruit & Vegetable Snacks",["fruit-snack", "dried-fruit", "fruit-bar",
                                  "vegetable-chip", "raisin", "date"]),
]

def assign_category(tags_str: str) -> str:
    if not isinstance(tags_str, str):
        return "Other"
    tags = tags_str.lower()
    for category, keywords in CATEGORY_RULES:
        if any(kw in tags for kw in keywords):
            return category
    return "Other"

df["primary_category"] = df["categories_tags"].apply(assign_category)

# Distribution
cat_counts = df["primary_category"].value_counts()
print(cat_counts)

fig = px.bar(
    cat_counts.reset_index(),
    x="primary_category", y="count",
    title="Product Count by Primary Category",
    color="primary_category",
    color_discrete_sequence=px.colors.qualitative.Bold,
    labels={"primary_category": "Category", "count": "# Products"},
)
fig.update_layout(showlegend=False, xaxis_tickangle=-30)
fig.show()

primary_category
Other                       43242
Cereals & Granola            8593
Dairy & Eggs                 5222
Candy & Confectionery        3347
Nuts & Seeds                 1825
Chips & Savory Snacks        1292
Protein Bars & Shakes         238
Fruit & Vegetable Snacks      186
Baked Goods & Cookies         112
Name: count, dtype: int64


## Story 3 · The Nutrient Matrix  Sugar vs Protein Scatter
The **"Empty Quadrant"** (top-left: High Protein + Low Sugar) is our Blue Ocean.

In [6]:
# Sample for scatter performance
SAMPLE = df.sample(min(15_000, len(df)), random_state=42).copy()

# Median splits
sugar_med   = SAMPLE["sugars_100g"].median()
protein_med = SAMPLE["proteins_100g"].median()

def quadrant(row):
    hi_p = row["proteins_100g"] >= protein_med
    lo_s = row["sugars_100g"]   <= sugar_med
    if hi_p and lo_s:     return "Blue Ocean (High P / Low S)"
    elif hi_p:            return "High Protein / High Sugar"
    elif lo_s:            return "Low Protein / Low Sugar"
    else:                 return "High Sugar / Low Protein"

SAMPLE["quadrant"] = SAMPLE.apply(quadrant, axis=1)

COLOR_MAP = {
    "Blue Ocean (High P / Low S)": "#00C49A",
    "High Protein / High Sugar"  : "#F59E0B",
    "Low Protein / Low Sugar"    : "#94A3B8",
    "High Sugar / Low Protein"   : "#EF4444",
}

fig = px.scatter(
    SAMPLE,
    x="sugars_100g", y="proteins_100g",
    color="quadrant",
    color_discrete_map=COLOR_MAP,
    hover_data=["product_name", "primary_category"],
    title="Sugar vs Protein — The Nutrient Matrix",
    labels={"sugars_100g": "Sugar (g/100g)", "proteins_100g": "Protein (g/100g)"},
    opacity=0.55,
    size_max=6,
)
# Quadrant lines
fig.add_vline(x=sugar_med,   line_dash="dash", line_color="gray", annotation_text=f"Median Sugar: {sugar_med:.1f}g")
fig.add_hline(y=protein_med, line_dash="dash", line_color="gray", annotation_text=f"Median Protein: {protein_med:.1f}g")
fig.update_layout(legend_title_text="Quadrant", height=550)
fig.show()
print(f"\n Blue Ocean products: {(SAMPLE['quadrant'] == 'Blue Ocean (High P / Low S)').sum():,}")


 Blue Ocean products: 4,517


## Story 4 · The Recommendation

In [7]:

blue_ocean = df[
    (df["proteins_100g"] >= df["proteins_100g"].median()) &
    (df["sugars_100g"]   <= df["sugars_100g"].median())
].copy()

# Best category in Blue Ocean quadrant
cat_bo = (
    blue_ocean.groupby("primary_category")
    .agg(
        n_products   = ("product_name",  "count"),
        avg_protein  = ("proteins_100g", "mean"),
        avg_sugar    = ("sugars_100g",   "mean"),
        avg_fiber    = ("fiber_100g",    "mean"),
    )
    .sort_values("n_products", ascending=False)
    .round(1)
)
display(cat_bo)

top_cat   = cat_bo.index[0]
top_prot  = cat_bo.loc[top_cat, "avg_protein"]
top_sugar = cat_bo.loc[top_cat, "avg_sugar"]

insight = (
    f"Based on the data, the biggest market opportunity is in **{top_cat}**, "
    f"specifically targeting products with ≥ {top_prot:.0f}g of protein "
    f"and less than {top_sugar:.0f}g of sugar per 100g."
)
print("\n" + "="*70)
print("KEY INSIGHT")
print("="*70)
print(insight)


,n_products,avg_protein,avg_sugar,avg_fiber
primary_category,,,,
Other,13778,16.40,1.40,3.00
Cereals & Granola,3058,10.30,1.90,4.80
Dairy & Eggs,1311,17.60,1.20,0.40
Nuts & Seeds,590,16.20,1.90,8.40
Chips & Savory Snacks,449,9.60,1.50,5.30
Candy & Confectionery,105,14.90,1.10,11.20
Protein Bars & Shakes,87,53.00,1.50,4.20
Baked Goods & Cookies,30,11.50,1.80,2.80
Fruit & Vegetable Snacks,4,22.20,1.10,14.10



KEY INSIGHT
Based on the data, the biggest market opportunity is in **Other**, specifically targeting products with ≥ 16g of protein and less than 1g of sugar per 100g.


## Bonus:  "Hidden Gem"  Top Protein Sources
We mine the `ingredients_text` of Blue Ocean products for the most common  
protein-rich ingredients using keyword frequency analysis.

In [8]:
PROTEIN_KEYWORDS = [
    "whey", "casein", "soy", "pea protein", "pea", "hemp",
    "peanut", "almond", "cashew", "egg", "milk protein",
    "chickpea", "lentil", "quinoa", "sunflower protein",
    "rice protein", "cricket", "collagen",
]

def extract_protein_sources(text: str) -> list:
    if not isinstance(text, str):
        return []
    text = text.lower()
    return [kw for kw in PROTEIN_KEYWORDS if kw in text]

blue_ocean_ingr = blue_ocean.dropna(subset=["ingredients_text"])
all_sources = []
for sources in blue_ocean_ingr["ingredients_text"].apply(extract_protein_sources):
    all_sources.extend(sources)

top_sources = Counter(all_sources).most_common(10)
src_df = pd.DataFrame(top_sources, columns=["Ingredient", "Frequency"])

fig = px.bar(
    src_df, x="Frequency", y="Ingredient",
    orientation="h",
    title="Top Protein Sources in Blue Ocean Products",
    color="Frequency",
    color_continuous_scale="Teal",
)
fig.update_layout(yaxis=dict(autorange="reversed"), coloraxis_showscale=False)
fig.show()

print("\nTop 3 Protein Sources:")
for i, (ing, freq) in enumerate(top_sources[:3], 1):
    print(f"  {i}. {ing.title()} — found in {freq} products")


Top 3 Protein Sources:
  1. Soy — found in 2753 products
  2. Whey — found in 757 products
  3. Egg — found in 686 products


## Candidate's Choice · Nutri-Score  Market Opportunity Matrix

**Why this matters:**  
The Nutri-Score (A–E) is the EU regulatory label that directly drives consumer purchase intent. A product can be High Protein / Low Sugar yet still score D or E (e.g., high sodium). We cross-reference the Blue Ocean quadrant with Nutri-Score to identify the *investable* white space products that are both nutritionally superior AND carry a favorable label.

> **Business implication:** Launching in a quadrant with High Protein +  
> Low Sugar + Nutri-Score A/B de-risks regulatory hurdles and differentiates  
> on-shelf in markets where front-of-pack labelling is mandatory (EU, UK, FR).

In [9]:
scored = df.dropna(subset=["nutriscore_grade"]).copy()
scored["nutriscore_grade"] = scored["nutriscore_grade"].str.upper()
scored = scored[scored["nutriscore_grade"].isin(["A","B","C","D","E"])]

# Average protein & sugar per (category × nutri-score)
heat = (
    scored.groupby(["primary_category", "nutriscore_grade"])
    .agg(
        avg_protein=("proteins_100g","mean"),
        n=("product_name","count")
    )
    .reset_index()
)

pivot = heat.pivot(index="primary_category", columns="nutriscore_grade", values="avg_protein").fillna(0)

fig = px.imshow(
    pivot,
    color_continuous_scale="RdYlGn",
    title="Avg Protein (g/100g) by Category × Nutri-Score<br><sup>Green = investable white space</sup>",
    labels={"color": "Avg Protein (g)"},
    aspect="auto",
)
fig.update_layout(height=450)
fig.show()

# Best combo
best_row = heat[heat["nutriscore_grade"].isin(["A","B"])].sort_values("avg_protein", ascending=False).head(3)
print("\n Top investable combos (High Protein + Good Nutri-Score):")
display(best_row[["primary_category","nutriscore_grade","avg_protein","n"]].reset_index(drop=True))


 Top investable combos (High Protein + Good Nutri-Score):


,primary_category,nutriscore_grade,avg_protein,n
0,Protein Bars & Shakes,A,20.78,4
1,Nuts & Seeds,A,17.95,343
2,Cereals & Granola,A,11.25,1190


In [10]:
!pip install streamlit pyngrok -q

In [15]:
from pyngrok import ngrok
import subprocess, threading, time
from google.colab import userdata

proc = subprocess.Popen(
    ["python", "-m", "streamlit", "run", "/content/drive/MyDrive/AmalitechProject/dashboard_pro.py",
     "--server.port=8501", "--server.headless=true"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(5)

ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))

ngrok.kill()

public_url = ngrok.connect(8501)
print(f"\n Dashboard accessible here :\n{public_url}")


 Dashboard accessible here :
NgrokTunnel: "https://plank-panama-prepay.ngrok-free.dev" -> "http://localhost:8501"


In [16]:
from google.colab import files

NOTEBOOK_NAME = "/content/drive/MyDrive/Colab Notebooks/sugar_trap_analysis.ipynb"

subprocess.run(
    [
        "jupyter",
        "nbconvert",
        "--to",
        "html",
        "--no-input",
        "--ExecutePreprocessor.enabled=False",
        NOTEBOOK_NAME,
    ],
    check=True,
)

html_name = NOTEBOOK_NAME.replace(".ipynb", ".html")
files.download(html_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>